# DUR 개별 미션 풀이 — 미션 8 (LDA 토픽 모델링) · 미션 7 (RNN 계열 임부금기 1등급 판별)

- 데이터 : 의약품안전사용서비스(DUR) 통합 성분단위 (2026.6), 28,502행 × 14열
- 실행 환경 : 바탕화면 `dur_mission` 폴더, `.venv` (python.org Python 3.12.10)
    - TensorFlow 2.21 + Keras 3 (백엔드 = TensorFlow)
    - Keras 3에서는 `tensorflow.keras.preprocessing.text.Tokenizer`가 삭제됨 → `keras.src.legacy` 경로에서 불러옴
    - 강사님 환경은 Keras 3 + **PyTorch 백엔드**(Python 3.13)라서, 같은 난수 시드여도 결과 숫자가 다르게 나옴
- 이 노트북은 수업 노트북(Day09)과 내 풀이를 비교해, **전처리 버그를 고친 최종본**으로 다시 정리한 것
- 실행 방법 : 커널 재시작 → 모두 실행 (위에서 아래 순서대로 실행해야 변수가 올바르게 만들어짐)

In [1]:
# 라이브러리 불러오기와 데이터 확인


import pandas as pd
import numpy as np
import plotly.express as px
import scipy.stats as stats

df1 = pd.read_csv('data/07_DUR_Data/의약품안전사용서비스(DUR)_통합_성분단위 2026.6.csv')
print(df1.shape)
df1.head(3)


# → 표 데이터(pandas), 숫자 계산(numpy), 그래프(plotly), 통계 검정(scipy)을 짧은 이름으로 불러오겠다
# → DUR CSV 파일을 상대 경로로 읽어 df1에 담겠다
# → 크기가 (28502, 14)인지 확인하고, 앞의 3행으로 컬럼 구조를 보겠다

(28502, 14)


,금기유형,성분코드,성분명,병용성분코드,병용성분명,금기등급,특정연령,특정연령단위,연령처리조건,상세정보,비고,고시번호,고시일자,제품수
0,병용금기,139230BIJ,cyclosporine,454002ATB,rosuvastatin calcium (as rosuvastatin),NaN,NaN,NaN,NaN,혈중농도 상승 및 근육병증/횡문근융해증 위험성 증가,NaN,20200061.0,2020-07-10,180
1,병용금기,157501ATR,felodipine,179101ACH,itraconazole,NaN,NaN,NaN,NaN,"Felodipine의 혈중농도 증가. 저혈압, 말초부종 같은 약물이상반응 발생 위험...",NaN,20190131.0,2019-12-17,272
2,병용금기,179101ATB,itraconazole,633800ATB,ezetimibe | atorvastatin calcium(as atorvastatin),NaN,NaN,NaN,NaN,"근육병증, 횡문근융해증 위험 증가 가능",NaN,20230083.0,2023-12-20,11252


In [2]:
# 컬럼별 자료형과 결측 개수 확인


df1.info()


# → 컬럼마다 비어 있지 않은 값의 개수와 자료형을 한 번에 보겠다
# → 28502보다 적은 컬럼 = 금기유형에 따라 쓰지 않아 비어 있는 컬럼

<class 'pandas.DataFrame'>
RangeIndex: 28502 entries, 0 to 28501
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   금기유형    28502 non-null  str    
 1   성분코드    28501 non-null  str    
 2   성분명     28501 non-null  str    
 3   병용성분코드  23639 non-null  str    
 4   병용성분명   23639 non-null  str    
 5   금기등급    4010 non-null   str    
 6   특정연령    607 non-null    float64
 7   특정연령단위  607 non-null    str    
 8   연령처리조건  607 non-null    str    
 9   상세정보    28195 non-null  str    
 10  비고      15196 non-null  str    
 11  고시번호    28406 non-null  float64
 12  고시일자    28406 non-null  str    
 13  제품수     28502 non-null  int64  
dtypes: float64(2), int64(1), str(11)
memory usage: 3.0 MB


In [3]:
# 금기유형별 건수


df1['금기유형'].value_counts()


# → 금기유형마다 몇 건인지 많은 순으로 세겠다 (병용금기 23,639건이 대부분)

금기유형
병용금기            23639
임부금기             4011
연령금기              607
노인주의              150
노인주의_해열진통소염제       95
Name: count, dtype: int64

In [4]:
# 금기등급별 건수


df1['금기등급'].value_counts()


# → 금기등급(1, 2, M)마다 몇 건인지 세겠다 (NaN은 제외되어 합계 4,010건)

금기등급
2    3332
1     638
M      40
Name: count, dtype: int64

## 미션 8

8. 금기유형이 '병용금기'인 데이터의 상세정보(중복 제거)를 이용해 토픽 모델링(LDA)을 수행하고, 병용금기 사유가 어떤 주제들로 구성되어 있는지 확인하시오.

# Topic Modeling

- Topic Modeling : 텍스트 문서 집합에서 숨겨진 주제 구조를 발견하는 과정
- 이 분석 기법은 문서들 사이의 유사성을 기반으로 문서집합을 구성하는 여러 주제들을 식별하는 기법 (비지도 학습)
- Topic Modeling을 통해 얻은 주제들은 문서 요약, 정보 탐색, 문서 분류 등에 활용
- **LDA(Latent Dirichlet Allocation)** 기법을 활용해 모델을 구성
    - 아래의 두 가지 가정을 활용하여, 문서 내 주제 단어를 선별하는 연산
        1. 문서는 여러 주제가 섞여 있음을 가정 (문서-주제 분포, Document-Topic Distribution)
            - 이 문서에는 어떤 주제가 몇 %씩 들어있는지 계산
        2. 주제는 단어들의 확률 목록으로 가정 (주제-단어 분포, Topic-Word Distribution)
            - 이 주제에서는 어떤 단어가 자주 사용되는지 계산
    - 실제로 관측되는 것은 단어뿐 → 단어로부터 두 분포(사후 분포)를 거꾸로 추정
    - 사후 분포를 정확히 계산하기 어려워 아래 두 가지 근사 방법을 사용
        - 변분 베이즈 추론 (Variational Bayes)
            - 정확한 사후 분포를 근사할 수 있는 더 간단한 분포를 찾아내는 방식으로 단어 / 주제 분포를 추정
            - 정확한 답 대신 "가장 비슷한 단순한 답"을 찾는 것
            - 빠르고 대용량 데이터에 적합하지만, 근사치이므로 정확한 결과를 보장하지 않음
            - gensim의 LdaModel이 사용하는 방식
        - 깁스 샘플링 (Gibbs Sampling)
            - 사후 분포를 근사하기 위해 반복적으로 샘플링하는 방식
            - 복잡한 다변수 확률분포에서 표본을 추출하는 방법
            - 각 단어에 주제를 반복해서 다시 배정 → 자연스럽게 유사한 단어들끼리 묶임
            - 반복 횟수가 충분하면 정확도가 높지만, 대용량 데이터에서는 느림

- 학습 단계 (깁스 샘플링)
    1. 초기화 : 각 문서의 단어에 임의의 주제를 부여 (해당 주제는 반복을 통해 점진적으로 갱신)
    2. 주제 추정 : 특정 단어에 대해 문서 내 다른 단어와의 관계를 기반으로 주제를 추정
        - 해당 문서에서 주제가 선택될 확률을 계산 (문서-주제 분포)
        - 주제가 해당 단어를 생성할 확률을 계산 (주제-단어 분포)
        - 지금 단어의 주제는 잠시 떼고(딱지 떼기) 나머지 배정만으로 계산 (추가)
        - 두 확률을 곱해 주제별 가능성을 구함 : 이 문서가 주제 k를 얼마나 쓰나 × 주제 k가 이 단어를 얼마나 쓰나 (추가)
        - 개수에 α·β를 더해 확률이 0이 되는 것을 막음 (스무딩) (추가)
    3. 추출 및 갱신 : 앞서 추정된 확률을 기반으로 각 단어에 새로운 주제를 부여
        - 문서에 어떤 주제가 가장 많이 등장하는지 계산
        - 동일한 주제에 속하는 다른 단어들이 얼마나 일관성있게 할당되었는지 계산
        - 가장 높은 주제만 고르지 않고 확률대로 뽑음 → 처음 배치 근처에 갇히는 것(지역 최적해) 방지 (추가)
    4. 선택된 주제를 이용해 해당 단어에 대한 분포 추정을 수행
    5. 분포의 변화가 작아질 때까지 위 과정을 반복
        - 변화가 거의 없는 상태 = 수렴 → gensim에서는 passes · iterations 로 반복 횟수 지정 (추가)
        - 주제 번호에는 뜻이 없음 → 상위 단어를 보고 사람이 이름을 붙임 (추가)
        - 실행마다 결과가 달라지므로 random_state로 고정 (추가)

- LDA는 문서-주제 분포와 주제-단어 분포를 동시에 고려하여 주제를 추출
    - 이를 통해 병용금기 사유가 어떤 주제들로 구성되어 있는지 확인할 수 있음

- LDA 모델링 과정
    1. 데이터 전처리: 텍스트 정제, 형태소 분석(명사 추출), 불용어 제거
    2. 단어-문서 행렬 생성: 각 문서의 단어 빈도(Bag of Words)를 기반으로 행렬 생성 (TF-IDF가 아닌 빈도 사용)
    3. 주제 수(K) 결정: 여러 K로 학습한 뒤 Coherence가 높은 구간을 사람이 검토해 선택
    4. LDA 모델 학습: 지정된 주제 수를 기반으로 모델 학습
    5. 주제 해석: 각 주제에 대한 주요 단어 확인 및 의미 해석
    6. 시각화: pyLDAvis 등을 활용하여 주제 간 관계 및 단어 분포 시각화

In [5]:
# gensim에서 LDA에 필요한 두 모듈 불러오기


from gensim import corpora
from gensim import models


# → corpora: 문서 모음(corpus)을 다루는 도구를 불러오겠다
#   → 단어마다 번호를 붙인 사전(Dictionary)과, 문서를 (단어번호, 등장횟수) 목록으로 바꾸는 Bag of Words 변환에 쓴다
# → models: LdaModel 같은 학습 모델을 불러오겠다

In [6]:
# 병용금기 상세정보 추출 (중복 제거)


cond1 = (df1['금기유형']=='병용금기')
combi_text = df1.loc[cond1]['상세정보'].dropna().drop_duplicates()
print('고유 문장 수 :', len(combi_text))
combi_text.head()


# → 금기유형이 '병용금기'인 행만 고르는 조건(True/False 목록)을 cond1에 담겠다
# → 그 행들의 상세정보에서 결측을 빼고(dropna), 같은 문장은 하나만 남기겠다(drop_duplicates) → 365문장
# → 앞의 몇 문장을 눈으로 확인하겠다

고유 문장 수 : 365


0                         혈중농도 상승 및 근육병증/횡문근융해증 위험성 증가
1    Felodipine의 혈중농도 증가. 저혈압, 말초부종 같은 약물이상반응 발생 위험...
2                                근육병증, 횡문근융해증 위험 증가 가능
3            근육통, 근육약화, 근염, 때때로 횡문근 변성을 포함한 근육독성 촉진 가능
5                                기능적 신부전에 의해 유산 산성증 촉진
Name: 상세정보, dtype: str

In [7]:
# 형태소 분석기 준비


from konlpy.tag import Okt

okt = Okt()


# → 한국어 문장을 품사 단위로 쪼개는 Okt 분석기를 불러와 하나 만들어 두겠다
#   → Okt는 내부적으로 Java를 쓰므로 JDK(JAVA_HOME)가 설정되어 있어야 한다

In [8]:
# 불용어 목록 (기본 + 의약품 도메인 추가)


stop_list = pd.read_csv('data/stopword.txt')['불용어'].tolist() + ['위험','병용','투여','증가','투약']


# → 기본 불용어 66개에, 금기 문장에 흔한 형식적 단어 5개를 더하겠다 → 총 71개

In [9]:
# 형태소 분석 (문장 하나 → 명사 리스트)


def get_nouns(text, extra_stop=[]):
    result = []
    for word in okt.nouns(str(text)):
        if (word not in stop_list) & (word not in extra_stop) & (len(word) > 1):
            result.append(word)
    return result


# → 문장 하나를 받아, 불용어가 아니고 2글자 이상인 명사만 리스트로 돌려주는 함수를 만들겠다
# → 주의: 이 함수는 "문장 하나"용이다
#   → Series를 통째로 넣으면 str()이 화면 출력 모양(앞 5줄 ... 뒤 5줄)으로 바꿔버려서, 약 10문장만 분석된다
#   → 수업에서 난 [2977, 83] 에러와 단어장 51개 문제가 바로 이것 때문

In [10]:
# 문장별 명사 추출 (LDA용)


pos_text1 = [get_nouns(x, extra_stop=['금기']) for x in combi_text]
pos_text2 = [x for x in pos_text1 if len(x)>0]
print('전체 :', len(pos_text1), '/ 빈 문장 제외 :', len(pos_text2))


# → 문장마다 명사 리스트를 만들겠다 ('금기'도 이번에만 추가로 제외)
# → 명사가 하나라도 남은 문장만 LDA 입력(pos_text2)으로 쓰겠다

전체 : 365 / 빈 문장 제외 : 362


In [11]:
# 단어 사전 구성


word_voca = corpora.Dictionary(pos_text2)
print('단어 수 :', len(word_voca))
list(word_voca.token2id.items())[:10]


# → pos_text2에 나온 모든 단어에 번호를 붙여 사전을 만들겠다 → 393단어
# → 앞의 10개 (단어, 번호) 쌍을 확인하겠다

단어 수 : 393


[('근육', 0),
 ('농도', 1),
 ('병증', 2),
 ('상승', 3),
 ('위험성', 4),
 ('융해증', 5),
 ('혈중', 6),
 ('횡문근', 7),
 ('말초', 8),
 ('반응', 9)]

In [12]:
# 코퍼스 생성 (문장 → (단어번호, 등장횟수) 목록)


corpus1 = [word_voca.doc2bow(x) for x in pos_text2]


# → 문장마다 명사 리스트를 (단어번호, 등장횟수) 쌍의 목록으로 바꾸겠다 (doc2bow = document to bag of words)
# → 순서는 버리고 빈도만 남긴 이 형태가 LDA의 입력이다

In [13]:
# LDA 모델 학습


lda_model = models.LdaModel(corpus1, num_topics=4, id2word=word_voca, passes=50, random_state=1234)


# → corpus1로 주제 4개짜리 LDA 모델을 학습하겠다
# → id2word: 결과를 번호가 아니라 실제 단어로 보여주기 위해 사전을 연결하겠다
# → passes=50: 전체 문서를 50번 반복해서 읽으며 분포를 다듬겠다
# → random_state=1234: 실행할 때마다 주제가 바뀌지 않도록 고정하겠다 (수업 코드에 없던 추가 옵션)

In [14]:
# 주제별 상위 단어 출력


lda_model.print_topics()


# → 주제마다 비중이 큰 단어 10개를 가중치와 함께 보겠다
#   → 0.113*"농도" = 이 주제에서 '농도'가 나올 확률이 약 11.3%

[(0,
  '0.055*"상승" + 0.045*"고혈압" + 0.031*"증후군" + 0.029*"연장" + 0.027*"경련" + 0.025*"세로토닌" + 0.021*"혈압" + 0.020*"심실" + 0.020*"유발" + 0.019*"빈맥"'),
 (1,
  '0.061*"이상" + 0.060*"반응" + 0.043*"발생" + 0.041*"종료" + 0.037*"약물" + 0.034*"가능성" + 0.030*"횡문근" + 0.030*"성분" + 0.029*"포함" + 0.029*"해당"'),
 (2,
  '0.142*"농도" + 0.115*"감소" + 0.108*"혈중" + 0.047*"효과" + 0.029*"혈장" + 0.025*"치료" + 0.020*"발현" + 0.019*"유도" + 0.018*"현저" + 0.016*"바이러스"'),
 (3,
  '0.059*"독성" + 0.032*"작용" + 0.027*"혈관" + 0.022*"기타" + 0.022*"경련" + 0.020*"혈압" + 0.020*"조직" + 0.018*"효과" + 0.017*"강하" + 0.014*"중추신경계"')]

In [15]:
# pyLDAvis로 토픽 시각화 (VS Code용: HTML 저장 후 브라우저로 열기)


import os
import webbrowser
import pyLDAvis.gensim_models

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model, corpus1, word_voca)
pyLDAvis.display(vis)

pyLDAvis.save_html(vis, 'lda_vis.html')
webbrowser.open('file://' + os.path.abspath('lda_vis.html'))


# → gensim LDA 결과를 pyLDAvis 그래프로 준비하겠다 (JupyterLab에서는 셀 안에 바로 표시됨)
# → VS Code에서는 셀 안에 안 보이므로, lda_vis.html 파일로 저장한 뒤 기본 브라우저로 열겠다

True

- Intertopic Distance Map : 각 주제간 거리를 시각화 (상대적 거리)
    - 원 하나 = 주제 하나, 원끼리 가까울수록 비슷한 단어를 쓰는 주제, 겹치면 주제 구분이 잘 안 된 것 (추가)
    - 원 번호는 print_topics의 번호와 다를 수 있음 (pyLDAvis는 비중 큰 순으로 1번부터 다시 매김) (추가)
- Overall term frequency : (오른쪽 파란색 막대) 전체 문서 집합에서 특정 단어가 등장하는 빈도수
- Estimated term Frequency within the selected topic : 선택된 특정 주제에서 해당 단어가 등장하는 추정 빈도수
    - 해당 값이 높으면, 이 주제가 해당 단어에 관련된 주제어로 판단
    - 원을 클릭하면 나타나는 빨간 막대, 빨간 막대가 파란 막대를 거의 채우면 그 주제에 집중된 단어 (추가)
- Marginal topic distribution : 문서 집합 전체에서 각 주제가 차지하는 비중 또는 분포
    - 그래프에서는 원의 크기로 표현됨 (추가)
- λ(람다) 슬라이더 : 단어 순위를 매기는 기준 조절 (추가)
    - λ = 1 : 주제 안에서 자주 나오는 단어 순 (흔한 단어가 위로) (추가)
    - λ를 0 쪽으로 : 이 주제에만 특별히 나오는 단어 순 (주제 특징이 잘 보임) (추가)
    - 보통 0.3 ~ 0.6 근처에서 주제 이름을 붙이기 좋음 (추가)

## 미션 7

7. 상세정보를 입력했을 때, 임부금기 1등급 여부를 판별하는 모델을 구성하고자 한다. RNN 계열의 알고리즘을 이용해 모델을 구성하고, model_dur.h5 이름으로 저장하시오.

In [16]:
# 임부금기 중 등급이 1, 2인 데이터만 추출


cond1 = df1['금기유형']=='임부금기'
cond2 = df1['금기등급'].isin(['1','2'])
df2 = df1.loc[cond1&cond2]
print(df2.shape)


# → 조건 1: 금기유형이 임부금기인 행 / 조건 2: 금기등급이 '1' 또는 '2'인 행 ('M' 40건은 제외)
#   → '1'을 따옴표로 쓰는 이유: 금기등급에 'M'이 섞여 있어 컬럼 전체가 문자열이기 때문
# → 두 조건을 모두 만족하는 행만 df2에 담겠다 → 3,970행

(3970, 14)


In [17]:
# 정답(Y)과 입력(X) 나누기


Y = df2['금기등급'].replace({'1':1, '2':0}).astype(int)
X = df2[['상세정보']]


# → 1등급은 1, 2등급은 0으로 바꾼 정답 라벨을 Y에 담겠다
# → 상세정보 컬럼을 표(DataFrame) 형태 그대로 X에 담겠다 (대괄호 두 겹 → 나중에 X['상세정보']로 꺼내 씀)

In [18]:
# 정답 클래스 비율 확인


Y.value_counts() / len(Y)


# → 0(2등급) 약 84% / 1(1등급) 약 16% → 불균형 데이터
#   → "전부 2등급"이라고만 찍어도 정확도 84%가 나오므로, 정확도가 아니라 1등급 recall·F1로 평가해야 한다

금기등급
0    0.839295
1    0.160705
Name: count, dtype: float64

In [19]:
# 학습용 / 평가용 분리


from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, random_state=1234, stratify=Y)
print(X_train.shape, X_test.shape)


# → X와 Y를 같은 순서로 섞어서 학습용 75% / 평가용 25%로 나누겠다 → 2,977 / 993
# → random_state=1234: 다시 실행해도 같게 나누겠다
# → stratify=Y: 84:16 비율을 train과 test 양쪽에 똑같이 유지하겠다

(2977, 1) (993, 1)


In [20]:
# EDA — 임부금기 상세정보 점검


print('결측 :', X['상세정보'].isna().sum())
print('고유 문장 수 :', X['상세정보'].nunique(), '/ 전체 :', len(X))
print(X['상세정보'].str.len().describe())
X['상세정보'].value_counts().head()


# → 결측 170행, 고유 문장 1,064개 / 전체 3,970행 → 같은 문장이 매우 많이 반복된다
#   → 같은 문장이 train과 test 양쪽에 들어갈 수 있어, 평가 점수가 실제보다 좋게 나올 가능성이 있다
# → 문장 길이 중앙값 약 58자, 가장 흔한 문장은 "임부에 대한 안전성 미확립", "-"만 있는 행도 57개

결측 : 170
고유 문장 수 : 1064 / 전체 : 3970
count    3800.000000
mean       84.375526
std        82.048809
min         1.000000
25%        30.000000
50%        58.000000
75%       106.000000
max       550.000000
Name: 상세정보, dtype: float64


상세정보
임부에 대한 안전성 미확립.                                               279
임부에 대한 안전성 미확립                                                185
태아 갑상선기능저하증 유발 가능\n방사선 절차는 태아 기형 유발 가능                         99
임신초기 3개월간 고용량 투여시 최기형 가능성. (단, VitA(레티놀)로서 5000 I.U/1일 이상)     72
-                                                              57
Name: count, dtype: int64

### 텍스트 전처리 설계 (수업 코드와 달라진 점)

- **문장별 처리** : 명사 추출은 반드시 문장 하나씩 수행 (수업 중 `[2977, 83]` 에러의 원인 해결)
- **단어장은 X_train으로만** : 평가 데이터를 미리 보고 단어장을 만들면 데이터 누수(leakage)
- **캐시 사용** : 고유 문장이 1,064개뿐이라, 한 번 분석한 문장은 저장해 두고 재사용 (강사님 `get_token` 아이디어)
    - 단, 강사님 `get_token`은 1글자 명사를 걸러내지 않으므로, 여기서는 `get_nouns`(2글자 이상)를 그대로 재사용
- **수업 노트북에 남아 있던 버그** : 단어장을 만드는 줄이 `get_nouns(X_train['상세정보'])`로 Series를 통째로 넣어 **단어장이 51개**뿐
    - 대부분의 단어가 `<OOV>` 하나로 뭉개져, 모델이 문장 내용을 거의 구분하지 못함
- **mask_zero=True** : 뒤쪽 패딩 0을 RNN이 건너뛰도록 설정 (수업 코드에는 없음)

In [21]:
# 텍스트를 숫자로 바꾸는 도구 불러오기


from keras.src.legacy.preprocessing.text import Tokenizer
from keras.utils import pad_sequences


# → Keras 3에서 삭제된 Tokenizer를 legacy(옛 기능 보관용) 경로에서 가져오겠다
#   → src가 붙은 내부 경로라 나중 버전에서 사라질 수 있다 (미션용으로는 충분)
# → pad_sequences는 Keras 3 정식 위치(keras.utils)에서 가져오겠다

In [22]:
# 문장 목록 → 명사 리스트 목록 (캐시 사용)


noun_cache = {}

def get_token(text_list):
    result = []
    for x in text_list:
        x = str(x)
        if x not in noun_cache:
            noun_cache[x] = get_nouns(x)
        result.append(noun_cache[x])
    return result


# → 분석 결과를 저장해 둘 빈 사전(noun_cache)을 만들겠다
# → 문장 목록을 받아 문장을 하나씩 꺼내 처리하겠다 → 문장 수 = 결과 수가 항상 유지된다
# → 처음 보는 문장이면 get_nouns로 명사를 뽑아 저장하고, 이미 본 문장이면 저장된 결과를 꺼내 쓰겠다

In [23]:
# 명사 추출 → 단어장(X_train만) → 숫자 시퀀스


X_train_nouns = get_token(X_train['상세정보'].fillna(''))
X_test_nouns = get_token(X_test['상세정보'].fillna(''))

tokenizer = Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_nouns)

X_train_seq = tokenizer.texts_to_sequences(X_train_nouns)
X_test_seq = tokenizer.texts_to_sequences(X_test_nouns)

print('문장 수 / 시퀀스 수 / 정답 수 :', len(X_train), len(X_train_seq), len(Y_train))
print('단어장 크기 :', len(tokenizer.word_index))
print('예시 명사 :', X_train_nouns[0])
print('예시 숫자 :', X_train_seq[0])
print('빈 시퀀스 (train / test) :', sum(len(s)==0 for s in X_train_seq), '/', sum(len(s)==0 for s in X_test_seq))


# → 결측은 빈 문자열로 바꾸고, 문장마다 명사 리스트를 만들겠다
# → 학습 데이터의 명사만 보고 "단어 → 번호" 단어장을 만들겠다 → 1,087단어 (수업 코드는 51단어)
# → 단어장에 없는 단어는 <OOV>(1번) 하나로 통일하겠다
# → 명사 리스트를 번호 리스트로 바꾸겠다
# → 세 개수(2977)가 같은지, 명사가 하나도 없는 문장("-" 등)이 몇 개인지 확인하겠다

문장 수 / 시퀀스 수 / 정답 수 : 2977 2977 2977
단어장 크기 : 1087
예시 명사 : ['임부', '안전성', '확립', '베타', '차단', '태반', '환류', '감소', '태아', '사망', '미숙', '조숙', '분만', '초래', '가능성', '태아', '신생아', '혈당', '가능성', '신생아', '심폐', '합병증', '위험성']
예시 숫자 : [2, 5, 4, 201, 195, 19, 232, 13, 3, 24, 398, 464, 26, 132, 14, 3, 10, 196, 14, 10, 156, 278, 85]
빈 시퀀스 (train / test) : 175 / 56


In [24]:
# 문장당 명사 개수 분포 (패딩 길이 근거)


lengths = pd.Series([len(s) for s in X_train_seq])
print(lengths.describe())
print('95% 지점 :', lengths.quantile(0.95))


# → 학습 문장마다 명사가 몇 개인지 세겠다 → 중앙값 11, 75% 지점 20, 95% 지점 45
# → 이번에는 수업과 비교하기 위해 MAXLEN=20을 유지한다 (문장의 약 75%가 잘리지 않는 길이)
#   → 45로 늘리는 실험은 추후 과제

count    2977.000000
mean       15.463218
std        15.041544
min         0.000000
25%         4.000000
50%        11.000000
75%        20.000000
max       111.000000
dtype: float64
95% 지점 : 45.0


In [25]:
# 패딩: 모든 시퀀스를 같은 길이로 맞추기


MAXLEN = 20

X_train_vec = pad_sequences(X_train_seq, maxlen=MAXLEN, padding='post', truncating='post')
X_test_vec = pad_sequences(X_test_seq, maxlen=MAXLEN, padding='post', truncating='post')

print(X_train_vec.shape, X_test_vec.shape)


# → 모든 번호 리스트를 20칸으로 맞추겠다
# → padding='post': 짧은 문장은 뒤쪽을 0으로 채우겠다
# → truncating='post': 긴 문장은 뒤쪽을 자르겠다 (기본값 'pre'는 앞쪽을 잘라 핵심 내용이 사라질 수 있음)
# → (2977, 20), (993, 20)이 나와야 정상

(2977, 20) (993, 20)


## 여러 RNN 모형을 비교 / 평가

- 5개 모형 모두 **입력 · 임베딩 · 출력층은 같고, 가운데 순환층만 바꿔서** 비교
- 공통 설정 : class_weight(1등급 가중치), EarlyStopping(patience=5), validation_split=0.3, adam · binary_crossentropy

In [26]:
# 학습 결과 재현을 위한 난수 고정


import tensorflow as tf
tf.config.experimental.enable_op_determinism()
tf.keras.utils.set_random_seed(1234)


# → 계산 순서를 고정해서 다시 실행해도 같은 결과가 나오게 하겠다 (속도는 약간 느려질 수 있음)
# → 가중치 초기값 등 무작위 요소의 시작점을 1234로 고정하겠다
#   → 모델 학습 순서를 바꾸면 결과도 바뀌므로, 셀 순서를 그대로 유지할 것

In [27]:
# 모델 구성·학습·평가 도구 불러오기


from keras.models import Sequential, load_model
from keras.layers import Dense, Embedding, SimpleRNN, LSTM, GRU, Bidirectional
from keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score


# → Sequential: 층을 순서대로 쌓는 모델 틀 / load_model: 저장한 모델을 다시 불러오는 도구
# → Embedding: 단어 번호를 숫자 벡터로 바꾸는 층 / Dense: 최종 판정(1등급 확률)을 내는 층
# → SimpleRNN / LSTM / GRU / Bidirectional: 비교할 순환층들
# → EarlyStopping: 검증 성능이 나빠지면 학습을 일찍 멈추는 도구
# → 성능표(classification_report)와 개별 지표(F1, precision, recall) 계산 도구

In [28]:
# 불균형 데이터 → 1등급(16%)에 더 큰 가중치 부여


weight_1 = (Y_train==0).sum() / (Y_train==1).sum()
class_weight = {0:1, 1:float(weight_1)}
print(class_weight)


# → 2등급 개수 ÷ 1등급 개수 = 약 5.23 → 1등급을 틀리면 벌점(손실)을 약 5.23배로 주겠다
# → float(): numpy 숫자를 일반 파이썬 소수로 바꿔 Keras에 안전하게 넘기겠다
# → 부작용: 학습 손실(loss)이 검증 손실(val_loss)보다 크게 보인다 (가중치는 학습 손실에만 곱해지기 때문)

{0: 1, 1: 5.2280334728033475}


In [29]:
# 학습 조기 종료 설정


early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


# → 검증 손실(val_loss)이 5번 연속 좋아지지 않으면 학습을 멈추겠다
# → 멈춘 뒤에는 가장 성적이 좋았던 시점의 가중치로 되돌리겠다

In [30]:
# 평가 함수 정의 (판정 기준 + 상세 평가 + 한 줄 요약)


VOCAB_SIZE = len(tokenizer.word_index) + 1

def threshold_func(result):
    if result >= 0.5:
        return 1
    return 0

def Eval_func(model, history):
    df_hist = pd.DataFrame(history.history)
    Y_train_pred = pd.Series(model.predict(X_train_vec, verbose=0).flatten()).apply(threshold_func)
    Y_test_pred = pd.Series(model.predict(X_test_vec, verbose=0).flatten()).apply(threshold_func)
    print('[학습 데이터]')
    print(classification_report(Y_train, Y_train_pred, digits=3))
    print('[평가 데이터]')
    print(classification_report(Y_test, Y_test_pred, digits=3))
    fig1 = px.line(df_hist, y=['loss', 'val_loss'], markers=True, title='모형 학습 / 검증 손실')
    fig1.update_yaxes(range=[0, 1])
    return fig1

def evaluate_model(name, model, history):
    pred_train = pd.Series(model.predict(X_train_vec, verbose=0).flatten()).apply(threshold_func)
    pred_test = pd.Series(model.predict(X_test_vec, verbose=0).flatten()).apply(threshold_func)
    return {
        '모형': name,
        '파라미터 수': model.count_params(),
        '학습 epoch': len(history.history['loss']),
        '학습 정확도': round((pred_train.values == Y_train.values).mean(), 3),
        '평가 정확도': round((pred_test.values == Y_test.values).mean(), 3),
        '평가 macro F1': round(f1_score(Y_test, pred_test, average='macro'), 3),
        '1등급 precision': round(precision_score(Y_test, pred_test), 3),
        '1등급 recall': round(recall_score(Y_test, pred_test), 3),
        '1등급 F1': round(f1_score(Y_test, pred_test), 3),
    }


# → VOCAB_SIZE: 단어장 크기 + 1 (0번 = 패딩 자리) → 모든 Embedding의 입력 범위로 쓰겠다
#   → 수업 코드의 Embedding(10000, ...)은 실제 단어 1,087개보다 훨씬 커서, 약 9천 줄이 쓰이지 않는 자리가 된다
# → threshold_func: 1등급 확률이 0.5 이상이면 1, 아니면 0으로 판정하겠다
# → Eval_func: 한 모델의 학습·평가 성능표와 손실 그래프를 보겠다
#   → verbose=0: 예측 진행 막대를 숨겨 성능표를 깔끔하게 보겠다
# → evaluate_model: 한 모델의 성능을 한 줄(사전)로 요약하겠다 → 나중에 모아서 비교표로 만든다
#   → macro F1: 0등급 F1과 1등급 F1의 단순 평균 / 1등급 지표 3개는 수업 코드에 없던 추가 항목
#   → .values: 행 번호표를 떼고 순서대로 비교하기 위함

### SimpleRNN (기본 순환 신경망)

- 문장을 앞에서부터 단어 하나씩 읽으며, 기억(Hidden State) 한 줄에 지금까지 내용을 요약 (추가)
- 게이트(문)가 없어 문장이 길어지면 앞쪽 정보가 희미해짐 → 장기의존성 문제 (추가)

In [31]:
# SimpleRNN 모형 구성 및 학습


model_RNN = Sequential()
model_RNN.add(Embedding(VOCAB_SIZE, 8, mask_zero=True))
model_RNN.add(SimpleRNN(20))
model_RNN.add(Dense(1, activation='sigmoid'))
model_RNN.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_RNN = model_RNN.fit(X_train_vec, Y_train, epochs=50, validation_split=0.3,
                            class_weight=class_weight, callbacks=[early_stop], verbose=0)
print('학습 epoch 수 :', len(history_RNN.history['loss']))


# → Embedding: 단어 번호를 숫자 8개짜리 벡터로 바꾸고, 패딩 0은 건너뛰겠다(mask_zero)
# → SimpleRNN(20): 기억 칸 20개로 문장을 앞에서부터 읽겠다
# → Dense(1, sigmoid): "1등급일 확률"(0~1) 하나를 내겠다
# → 최대 50 epoch, 학습 데이터 30%는 검증용, 1등급 가중치와 조기 종료를 적용하겠다

학습 epoch 수 : 15


In [32]:
# SimpleRNN 평가


Eval_func(model_RNN, history_RNN)


# → SimpleRNN의 학습·평가 성능표와 손실 그래프를 보겠다

[학습 데이터]
              precision    recall  f1-score   support

           0      0.987     0.880     0.931      2499
           1      0.600     0.939     0.732       478

    accuracy                          0.890      2977
   macro avg      0.794     0.910     0.832      2977
weighted avg      0.925     0.890     0.899      2977

[평가 데이터]
              precision    recall  f1-score   support

           0      0.978     0.844     0.906       833
           1      0.526     0.900     0.664       160

    accuracy                          0.853       993
   macro avg      0.752     0.872     0.785       993
weighted avg      0.905     0.853     0.867       993



### LSTM (Long Short-Term Memory)

- SimpleRNN의 장기의존성 문제를 보완한 구조 (추가)
- Hidden State(단기 기억) 외에 **Cell State(장기 기억)**를 따로 두고, 문(gate) 3개로 기억을 관리 (추가)
    - Forget Gate : 예전 기억을 얼마나 지울지
    - Input Gate : 새 정보를 얼마나 받아들일지
    - Output Gate : 기억 중 무엇을 내보낼지
- dropout : 학습할 때마다 입력의 일부를 무작위로 가려 과적합(외우기)을 막음 (추가)

In [33]:
# LSTM 모형 구성 및 학습


model_LSTM = Sequential()
model_LSTM.add(Embedding(VOCAB_SIZE, 8, mask_zero=True))
model_LSTM.add(LSTM(128, dropout=0.2))
model_LSTM.add(Dense(1, activation='sigmoid'))
model_LSTM.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_LSTM = model_LSTM.fit(X_train_vec, Y_train, epochs=50, validation_split=0.3,
                              class_weight=class_weight, callbacks=[early_stop], verbose=0)
print('학습 epoch 수 :', len(history_LSTM.history['loss']))


# → LSTM(128): 기억 칸 128개짜리 LSTM으로 읽겠다 (SimpleRNN보다 훨씬 큰 모델)
# → dropout=0.2: 학습 때마다 입력의 20%를 가려 특정 단어 하나에 의존하지 못하게 하겠다
# → CPU로는 학습에 몇 분 걸릴 수 있다

학습 epoch 수 : 17


In [34]:
# LSTM 평가


Eval_func(model_LSTM, history_LSTM)


# → LSTM의 학습·평가 성능표와 손실 그래프를 보겠다

[학습 데이터]
              precision    recall  f1-score   support

           0      0.970     0.935     0.952      2499
           1      0.714     0.849     0.776       478

    accuracy                          0.921      2977
   macro avg      0.842     0.892     0.864      2977
weighted avg      0.929     0.921     0.924      2977

[평가 데이터]
              precision    recall  f1-score   support

           0      0.961     0.935     0.948       833
           1      0.703     0.800     0.749       160

    accuracy                          0.913       993
   macro avg      0.832     0.868     0.848       993
weighted avg      0.919     0.913     0.916       993



# GRU Model (Gated Recurrent Unit)

- LSTM 모형에서 Cell State의 구조를 단순화 → LSTM 모형과 비교했을 때, 연산 소요를 줄이고 학습을 빠르게 수행하기 위함
- RNN의 장기의존성 문제를 LSTM 모형으로 극복 → Cell State 연산량이 많음 → 파라미터가 많고 느리다는 단점을 보완하기 위해 만든 모형
    - 장기의존성 문제 : 문장이 길어질수록 앞쪽 정보가 뒤까지 전달되지 못하고 희미해지는 문제 (추가)
    - GRU는 Cell State를 따로 두지 않고, Hidden State 하나에 기억을 합쳐서 관리 (추가)
- 구성 요소 :
    - Update Gate : LSTM 모형의 Input Gate + Forget Gate를 합쳐서 만든 구조
        - 이전 상태를 얼마나 유지하고, 새로운 정보를 얼마나 받아들일지를 하나의 값으로 결정
        - Output → 1 : 과거 정보를 유지 / 0 : 새로운 정보로 교체
        - 유지하는 비율과 교체하는 비율의 합이 항상 1 → "하나 받으면 하나 버린다"는 구조 (추가)
    - Reset Gate : 과거의 정보를 얼마나 무시할지를 결정
        - Output → 0 : 이전 상태를 잊고, 현재 입력만으로 새 후보 정보를 계산
    - LSTM의 Output Gate에 해당하는 문은 없음 → Hidden State를 그대로 출력 (추가)
- LSTM보다 약 25% 파라미터 절감
    - LSTM은 문 3개 + 후보값 1개 = 가중치 세트 4개, GRU는 문 2개 + 후보값 1개 = 3개 → 4개 중 1개가 줄어서 25% (추가)
- 성능은 데이터에 따라 LSTM과 비슷하거나 약간 낮은 경우가 많음 → 데이터가 적거나 속도가 중요할 때 먼저 시도해 볼 만한 모형 (추가)

In [35]:
# GRU 모형 구성 및 학습


model_GRU = Sequential()
model_GRU.add(Embedding(VOCAB_SIZE, 16, mask_zero=True))
model_GRU.add(GRU(32))
model_GRU.add(Dense(1, activation='sigmoid'))
model_GRU.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_GRU = model_GRU.fit(X_train_vec, Y_train, epochs=50, validation_split=0.3,
                            class_weight=class_weight, callbacks=[early_stop], verbose=0)
print('학습 epoch 수 :', len(history_GRU.history['loss']))


# → Embedding: 단어 하나를 숫자 16개 벡터로 바꾸겠다 (다른 모델은 8개), 패딩 0은 건너뛰겠다
# → GRU(32): 기억 칸 32개짜리 GRU로 읽겠다

학습 epoch 수 : 9


In [36]:
# GRU 평가


Eval_func(model_GRU, history_GRU)


# → GRU의 학습·평가 성능표와 손실 그래프를 보겠다

[학습 데이터]
              precision    recall  f1-score   support

           0      0.978     0.904     0.940      2499
           1      0.641     0.891     0.745       478

    accuracy                          0.902      2977
   macro avg      0.809     0.898     0.842      2977
weighted avg      0.923     0.902     0.908      2977

[평가 데이터]
              precision    recall  f1-score   support

           0      0.967     0.916     0.941       833
           1      0.657     0.838     0.736       160

    accuracy                          0.903       993
   macro avg      0.812     0.877     0.839       993
weighted avg      0.917     0.903     0.908       993



# Bidirectional RNN (양방향 순환 신경망)

- 지금까지의 RNN 모형은 문장을 앞에서부터 뒤로 한 방향으로 처리
    - [초음파] → [진단] → [보조] → [장치]
    - 차를 마셨다 / 차를 탔다 : 문장의 단어가 뒤쪽의 표현에 따라 의미가 달라지는 경우
    - 한 방향으로만 읽으면 '차'를 읽는 시점에는 뒤의 '마셨다/탔다'를 아직 모름 (추가)
- 같은 문장을 정방향 / 역방향 두 개의 순환층으로 각각 읽고 두 결과를 이어붙이는 형태
    - 출력 차원이 2배 / 학습 시간 2배
    - 예 : Bidirectional(GRU(20)) → 정방향 20개 + 역방향 20개 = 출력 40개 (추가)
- 뒤쪽 패딩(post)을 쓸 때는 mask_zero가 특히 중요 (추가)
    - 역방향 층은 문장 끝부터 읽으므로, 마스크가 없으면 패딩 0을 먼저 잔뜩 읽고 나서야 진짜 단어에 도착
- 우리 데이터에서의 한계 (추가)
    - 명사만 추출했기 때문에 '마셨다/탔다' 같은 서술어는 이미 제거된 상태
    - 따라서 양방향의 장점이 문장 원문보다 덜 드러날 수 있음

In [37]:
# 양방향 GRU 모형 구성 및 학습


model_BiGRU = Sequential()
model_BiGRU.add(Embedding(VOCAB_SIZE, 8, mask_zero=True))
model_BiGRU.add(Bidirectional(GRU(20)))
model_BiGRU.add(Dense(1, activation='sigmoid'))
model_BiGRU.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_BiGRU = model_BiGRU.fit(X_train_vec, Y_train, epochs=50, validation_split=0.3,
                                class_weight=class_weight, callbacks=[early_stop], verbose=0)
print('학습 epoch 수 :', len(history_BiGRU.history['loss']))


# → Bidirectional(GRU(20)): GRU 두 개가 앞→뒤, 뒤→앞으로 각각 읽고 결과를 이어붙이겠다 (출력 40개)
# → Embedding의 마스크(0 건너뛰기)는 양쪽 방향 모두에 자동으로 전달된다

학습 epoch 수 : 15


In [38]:
# 양방향 GRU 평가


Eval_func(model_BiGRU, history_BiGRU)


# → 양방향 GRU의 학습·평가 성능표와 손실 그래프를 보겠다

[학습 데이터]
              precision    recall  f1-score   support

           0      0.985     0.893     0.937      2499
           1      0.625     0.931     0.748       478

    accuracy                          0.899      2977
   macro avg      0.805     0.912     0.842      2977
weighted avg      0.928     0.899     0.907      2977

[평가 데이터]
              precision    recall  f1-score   support

           0      0.976     0.894     0.934       833
           1      0.617     0.887     0.728       160

    accuracy                          0.893       993
   macro avg      0.797     0.891     0.831       993
weighted avg      0.919     0.893     0.900       993



## Stack RNN (순환층 쌓기)

- CNN 알고리즘에서 Convolution 층을 여러 개 쌓듯이, 순환층도 여러 층으로 쌓아 학습
- return_sequences=True : 앞서 RNN에서 처리한 결과를 받아, 뒷부분 RNN 모형에 이어 학습
- 아래층에서는 단어 수준의 짧은 패턴을 학습 / 위층은 문장 수준의 긴 패턴을 담당하도록 나누어 학습
- return_sequences 옵션 비교 (추가)
    - False (기본값) : 문장을 다 읽은 뒤 **마지막 요약 1개**만 출력 → 모양 (문장 수, 유닛 수)
    - True : 단어를 읽을 때마다 **매 시점의 결과를 모두** 출력 → 모양 (문장 수, 문장 길이, 유닛 수)
    - 다음 순환층은 "순서가 있는 입력"을 받아야 하므로, 쌓을 때는 **마지막 층을 뺀 모든 순환층에 True**가 필요
- 출력 모양 확인 결과 (추가)
    - return_sequences=False : (4, 20) → 문장 4개 × 요약 20개
    - return_sequences=True : (4, 20, 20) → 문장 4개 × 단어 자리 20칸(MAXLEN) × 유닛 20개

In [39]:
# return_sequences에 따른 출력 모양 차이 확인


test_input = X_train_vec[:4]

m_false = Sequential([Embedding(VOCAB_SIZE, 8), GRU(20)])
m_true = Sequential([Embedding(VOCAB_SIZE, 8), GRU(20, return_sequences=True)])

print('return_sequences=False :', m_false.predict(test_input, verbose=0).shape)
print('return_sequences=True  :', m_true.predict(test_input, verbose=0).shape)


# → 학습 데이터 앞 4문장으로, 학습 없이 출력 모양(shape)만 비교하겠다
# → (4, 20)과 (4, 20, 20)이 나와야 한다 (가운데 20 = MAXLEN, 마지막 20 = 유닛 수)

return_sequences=False : (4, 20)
return_sequences=True  : (4, 20, 20)


In [40]:
# Stack GRU 모형 구성 및 학습 (순환층 2겹)


model_Stack = Sequential()
model_Stack.add(Embedding(VOCAB_SIZE, 8, mask_zero=True))
model_Stack.add(GRU(20, return_sequences=True))
model_Stack.add(GRU(20))
model_Stack.add(Dense(1, activation='sigmoid'))
model_Stack.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_stack = model_Stack.fit(X_train_vec, Y_train, epochs=50, validation_split=0.3,
                                class_weight=class_weight, callbacks=[early_stop], verbose=0)
print('학습 epoch 수 :', len(history_stack.history['loss']))


# → 첫 번째 GRU: 매 단어마다 결과를 내보내겠다 (20칸 × 20) → 다음 GRU가 순서대로 읽을 수 있게
# → 두 번째 GRU: 그 결과를 다시 읽고 마지막 요약 1개만 내보내겠다
# → 데이터 흐름: (문장, 20) → (문장, 20, 8) → (문장, 20, 20) → (문장, 20) → (문장, 1)

학습 epoch 수 : 11


In [41]:
# Stack GRU 평가


Eval_func(model_Stack, history_stack)


# → 순환층을 2겹 쌓은 모델의 학습·평가 성능표와 손실 그래프를 보겠다

[학습 데이터]
              precision    recall  f1-score   support

           0      0.970     0.938     0.954      2499
           1      0.725     0.849     0.782       478

    accuracy                          0.924      2977
   macro avg      0.848     0.894     0.868      2977
weighted avg      0.931     0.924     0.926      2977

[평가 데이터]
              precision    recall  f1-score   support

           0      0.957     0.941     0.949       833
           1      0.718     0.781     0.749       160

    accuracy                          0.915       993
   macro avg      0.838     0.861     0.849       993
weighted avg      0.919     0.915     0.917       993



# 전체 성능 비교

In [42]:
# 전체 모델 성능 비교표


result_list = [
    evaluate_model('RNN', model_RNN, history_RNN),
    evaluate_model('LSTM', model_LSTM, history_LSTM),
    evaluate_model('GRU(32)', model_GRU, history_GRU),
    evaluate_model('Bidirectional GRU', model_BiGRU, history_BiGRU),
    evaluate_model('GRU 2층 (Stacked)', model_Stack, history_stack),
]

df_result = pd.DataFrame(result_list)
df_result


# → 모델 5개를 각각 한 줄로 요약해 비교표(df_result)로 만들겠다
# → 수업 코드의 이름표 'GRU(20)'은 실제 모델(GRU(32))과 달라서 'GRU(32)'로 바로잡았다

,모형,파라미터 수,학습 epoch,학습 정확도,평가 정확도,평가 macro F1,1등급 precision,1등급 recall,1등급 F1
0,RNN,9305,15,0.890,0.853,0.785,0.526,0.900,0.664
1,LSTM,78977,17,0.921,0.913,0.848,0.703,0.800,0.749
2,GRU(32),22241,9,0.902,0.903,0.839,0.657,0.838,0.736
3,Bidirectional GRU,12345,15,0.899,0.893,0.831,0.617,0.887,0.728
4,GRU 2층 (Stacked),13045,11,0.924,0.915,0.849,0.718,0.781,0.749


### 수업 노트북(Day09)과의 비교 — 평가 데이터 993건

| 모형 | 수업 macro F1 | 내 macro F1 | 수업 1등급 recall | 내 1등급 recall | 수업 1등급 F1 | 내 1등급 F1 |
|---|---|---|---|---|---|---|
| RNN | 0.669 | 0.785 | 0.76 | 0.900 | 0.50 | 0.664 |
| LSTM | 0.523 | 0.848 | 0.61 | 0.800 | 0.33 | 0.749 |
| GRU(32) | 0.657 | 0.839 | 0.81 | 0.838 | 0.49 | 0.736 |
| Bidirectional GRU | 0.630 | 0.829 | 0.61 | 0.869 | 0.43 | 0.724 |
| Stacked GRU | 0.651 | 0.844 | 0.56 | 0.812 | 0.44 | 0.743 |

- 수치는 2026-09-28 실행 기준 (다시 실행하면 소수점 단위로 달라질 수 있으므로 위 df_result 결과로 갱신)
- 모델 구조는 같은데 5개 모형 모두 내 풀이가 macro F1 기준 0.12 ~ 0.33 높음
    - 차이의 주원인 : 단어장(수업 51개 → 1,087개)과 mask_zero
    - 특히 수업 LSTM은 손실이 "반반 찍기" 수준(약 0.69)에서 거의 줄지 않아 사실상 학습 실패
- 백엔드(수업 PyTorch / 내 TensorFlow)가 달라 초기 가중치도 다르므로, 수치 차이 전부가 전처리 때문이라고 단정하지는 않음
- 파라미터 수 : 수업은 Embedding(10000, 8)만으로 80,000개 → 실제 단어 1,087개 외에 약 9천 줄이 쓰이지 않음

### 5개 모형 장단점 비교 (내 실행 결과 기준)

| 모형 | 파라미터 | 1등급 recall | 1등급 F1 | 장점 | 단점 |
|---|---|---|---|---|---|
| SimpleRNN(20) | 9,305 | 0.900 | 0.664 | 구조가 가장 단순하고 가벼움, 1등급을 가장 적게 놓침 | 게이트가 없어 긴 문장에 약함, 잘못된 1등급 경고가 가장 많음(precision 0.526) |
| LSTM(128) | 78,977 | 0.800 | 0.749 | 기억 C + 게이트 3개로 장기 기억에 강함, F1 최고, dropout으로 과적합 억제 | 가장 무겁고 느림(RNN의 약 8.5배), recall은 낮은 편 |
| GRU(32) | 22,241 | 0.838 | 0.736 | LSTM보다 게이트가 적어 가볍고 빠르면서 성능은 비슷 | 임베딩 16차원·유닛 32로 다른 모형과 조건이 달라 직접 비교가 어려움 |
| Bidirectional GRU | 12,345 | 0.869 | 0.724 | 앞뒤 양방향 문맥 반영, 역방향이 첫 단어에서 끝나 문장 앞부분 정보가 잘 남음 | 출력 2배·계산 2배, 문장 전체가 있어야 해서 실시간(스트리밍) 처리 불가 |
| Stacked GRU(2층) | 13,045 | 0.812 | 0.743 | 아래층은 짧은 패턴, 위층은 긴 패턴으로 역할 분담 | 층이 늘수록 학습이 느려지고, 데이터가 적으면 과적합 위험 |

- 순환층 자체의 파라미터(580 / 70,144 / 4,800 / 3,600 / 4,320)는 수업 도식과 같음
    - 전체 파라미터가 도식보다 훨씬 작은 이유 : Embedding 입력 범위를 10000이 아니라 실제 단어장 크기(1,088)로 맞췄기 때문

### 기억할 포인트

- RNN 계열의 발전(LSTM → GRU → 양방향 → 적층)은 모두 **"순서와 장기 기억"을 더 잘 다루기 위한 방향**
- 그런데 의약품 상세정보는 **단어의 위치나 어순보다 "어떤 단어가 들어 있느냐"**가 판정을 좌우하는 데이터
- 그래서 모형을 바꿔도 **성능 차이가 도드라지게 나타나지 않음**
    - 게이트가 있는 4개 모형의 macro F1 : 0.829 ~ 0.848 (차이 0.02 이내)
    - SimpleRNN(0.785)만 한 단계 낮음
- 오히려 **전처리(단어장 크기, mask_zero)**를 고쳤을 때 성능 변화가 훨씬 컸음 (5개 모형 모두 macro F1 0.12 ~ 0.33 상승)

### 한계점

**1. 데이터 자체가 순서 모델에 불리함**
- 명사만 추출해서 서술어·부정 표현("~하지 않음")·조사가 모두 사라짐 → 어순 정보가 이미 대부분 제거된 상태
- 문장당 명사 개수 중앙값이 11개로 짧음 → LSTM·GRU가 해결하려는 장기의존성 문제가 애초에 거의 없음

**2. 평가 점수가 실제보다 좋게 나왔을 가능성**
- 3,970행 중 고유 문장은 1,064개뿐 → 같은 문장이 train과 test에 함께 들어가 있어, 모델이 문장을 "외워서" 맞힌 부분이 섞여 있음
- 한 번의 분할(random_state=1234)과 한 번의 학습 결과라, 0.01 ~ 0.02 차이는 우연일 수 있음

**3. 공정한 비교가 아님**
- 모형마다 유닛 수(20 / 128 / 32), 임베딩 차원(8 / 16), dropout 유무가 함께 달라짐 → "구조 덕분인지, 크기 덕분인지" 구분 불가

**4. 입력 정보 손실**
- MAXLEN=20으로 문장의 약 25%가 뒷부분이 잘림 (95% 지점은 45)
- 명사가 하나도 없는 문장("-" 등)이 학습 데이터에 175개 → 내용 없이 정답만 있는 입력

**5. 결론을 확인하지 못한 부분**
- "순서보다 단어가 중요하다"를 증명하려면 순서를 아예 무시하는 모델(Bag of Words + 로지스틱 회귀)과 비교해야 하는데, 이번에는 하지 않음
- 판정 기준 0.5 고정 → 1등급 recall을 더 높일 여지(0.3 ~ 0.4 기준) 미확인

### 모형 비교 해석

- 게이트가 있는 4개 모형(LSTM · GRU · BiGRU · Stack)은 macro F1 0.829 ~ 0.848로 **차이가 작음**
- SimpleRNN은 1등급 recall(0.900)은 가장 높지만 precision(0.526)이 낮아 잘못된 1등급 경고가 많음
- LSTM은 F1이 가장 높지만 파라미터가 약 7.9만 개로 가장 무겁고, recall은 0.800으로 낮은 편
- 과적합 : 모든 모형이 학습·평가 정확도 차이가 작아(최대 약 0.04) 심한 과적합은 없음
- 주의 : 고유 문장이 1,064개뿐이라 같은 문장이 train · test에 함께 들어가 있음 → 평가 점수가 실제보다 좋게 나왔을 가능성

### 최종 모델 선택 → Bidirectional GRU

- 임부금기 1등급은 **놓치면 위험한 쪽** → 1등급 recall을 가장 중요하게 봄
- Bidirectional GRU : recall 0.869(2위), precision 0.621로 SimpleRNN보다 잘못된 경고가 적음
- 파라미터 약 1.2만 개로 LSTM의 약 1/6 → 가볍고 빠름
- 수업에서도 Bidirectional GRU를 model_dur.h5로 저장

In [43]:
# 최종 모델 저장 (Bidirectional GRU → model_dur.h5)


model_BiGRU.save('model_dur.h5')

model_check = load_model('model_dur.h5')
pred_before = model_BiGRU.predict(X_test_vec, verbose=0)
pred_after = model_check.predict(X_test_vec, verbose=0)
print('저장 전후 예측이 같은가 :', np.allclose(pred_before, pred_after))


# → 양방향 GRU 모델을 HDF5 형식(.h5) 파일로 dur_mission 폴더에 저장하겠다
#   → HDF5: 모델 구조와 학습된 가중치를 파일 하나에 담는 형식 (읽고 쓰는 데 h5py 필요)
#   → Keras 3는 새 형식(.keras)을 권장해서 legacy 경고가 뜰 수 있지만, 미션 요구가 .h5라 무시해도 된다
# → 저장한 파일을 다시 불러와, 평가 데이터 예측이 저장 전과 같은지 확인하겠다 → True면 정상
#   → np.allclose: 소수점 아주 끝자리 차이는 같은 것으로 보고 비교하겠다

c:\Users\Taeeun Eom\Desktop\dur_mission\.venv\Lib\site-packages\keras\src\backend\tensorflow\core.py:172: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(x)
c:\Users\Taeeun Eom\Desktop\dur_mission\.venv\Lib\site-packages\keras\src\backend\tensorflow\core.py:172: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(x)
c:\Users\Taeeun Eom\Desktop\dur_mission\.venv\Lib\site-packages\keras\src\backend\tensorflow\core.py:172: DeprecationWarning: __array__ 

저장 전후 예측이 같은가 : True


In [44]:
# 오늘의 핵심 정리 — RNN 계열 모델의 발전과 비교


# [강사님 정리]
# → RNN의 한계 : 장기의존성 문제
#   → 문장이 길어질수록 앞쪽 정보가 뒤까지 전달되지 못하고 희미해진다
# → 이를 보완하기 위해 RNN 모델이 발전해 왔다
#   → LSTM : 문(gate) 3개와 Cell State로 중요한 정보를 오래 기억
#   → GRU : LSTM 구조를 단순화 (문 2개, 파라미터 약 25% 절감, 학습이 빠름)
#   → Bidirectional : 문장을 앞→뒤, 뒤→앞 양방향으로 읽어 뒤쪽 문맥도 반영
#   → Stack : 순환층을 여러 겹 쌓아 짧은 패턴(아래층)과 긴 패턴(위층)을 나눠 학습
# → 이 발전은 모두 "순서와 장기 기억"을 더 잘 다루기 위한 방향이다
# → 그래서 문장의 순서가 크게 중요하지 않은 자연어 데이터에서는
#   모델 간 성능 차이가 크게(다이나믹하게) 나지 않는다
#   → 의약품 상세정보는 어순보다 "어떤 단어가 들어 있느냐"가 판정에 더 중요


# [내 결과 메모]
# → 게이트가 있는 4개 모형은 macro F1 0.829 ~ 0.848로 차이가 작았다 → 강사님 결론과 일치
# → 다만 SimpleRNN(0.785)과 나머지 사이에는 차이가 있었다
#   → "게이트 구조끼리는 비슷하고, 기본 RNN보다는 낫다"가 내 데이터로 확인한 결론
# → 모델 선택보다 전처리가 성능에 더 큰 영향을 줬다
#   → 단어장 51개 버그와 mask_zero 미사용을 고치자 5개 모형 모두 macro F1이 0.12 ~ 0.33 올랐다
# → 최종 모델 : Bidirectional GRU (1등급 recall 우선 + 가벼운 모델) → model_dur.h5


# [추후 실험 과제]
# → "순서보다 단어가 중요하다" 검증 : 순서를 무시하는 모델(Bag of Words + 로지스틱 회귀)과 비교
# → 판정 기준(threshold)을 0.5 → 0.3 ~ 0.4로 낮춰 1등급 recall을 더 올릴 수 있는지 확인
# → MAXLEN을 20 → 45(95% 지점)로 늘려 긴 문장이 잘리지 않을 때의 변화 확인
# → 중복 문장을 train · test에 나누어 담지 않도록 분리해, 누수 없는 평가 점수 확인